# Class 3 (bonus) — RAG with LangChain & Vector Databases

**Week 6: Foundations of RAG and Chatbots** · companion to `class-3.ipynb`

The main Class 3 notebook builds RAG **from scratch** (a Python list of vectors). This notebook is the same pipeline with **LangChain** and real vector databases.

You will:
- Split the same style of policy docs into chunks
- Embed locally (`all-MiniLM-L6-v2` — Groq has no public embedding model)
- Store vectors in **Chroma**, retrieve, then generate with Groq
- Swap the same chunks into **FAISS** (same job, different vector DB)
- Optionally attach the retriever as a LangChain **tool** (the Week 5 agent pattern, now over documents)

Run cells in order with **Shift+Enter**. Generation cells need a `GROQ_API_KEY` (Colab secret or env var). Indexing and retrieval run without a key.

## Setup

```bash
export GROQ_API_KEY="gsk-..."
```

In Colab: add a secret named `GROQ_API_KEY` and enable notebook access.

In [ ]:
!pip install -q langchain langchain-groq langchain-chroma langchain-huggingface langchain-text-splitters langchain-community sentence-transformers faiss-cpu

In [ ]:
import os

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
try:
    from google.colab import userdata
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY")
except Exception:
    pass

if not GROQ_API_KEY:
    print(
        "No GROQ_API_KEY found.\n"
        "Set it in your environment or add a Colab secret named GROQ_API_KEY.\n"
        "Indexing and retrieval still run. Generation cells will skip."
    )
else:
    print("Found GROQ_API_KEY. RAG generation cells are ready.")

## 1. The same policy docs as Class 3

These three policies match the from-scratch Class 3 notebook, so you can compare a Python list of vectors with Chroma and FAISS.

In [ ]:
from langchain_core.documents import Document

raw_documents = [
    Document(
        page_content=(
            "Vacation Policy: New hires accrue 15 paid vacation days per year, credited monthly starting from their first day. "
            "Unused days roll over up to a maximum of 5 days into the next year."
        ),
        metadata={"source": "vacation.md"},
    ),
    Document(
        page_content=(
            "Expense Policy: Employees must submit expense reports within 30 days of purchase using the finance portal. "
            "Reports missing a receipt over $25 will be returned for correction."
        ),
        metadata={"source": "expenses.md"},
    ),
    Document(
        page_content=(
            "Remote Work Policy: Remote employees must be reachable by chat or phone between 10am and 3pm in their local time zone, "
            "and are expected to attend the weekly all-hands meeting on video."
        ),
        metadata={"source": "remote.md"},
    ),
]

for d in raw_documents:
    print(f"[{d.metadata['source']}] {d.page_content[:88]}...")

## 2. Chunk the documents

LangChain's `RecursiveCharacterTextSplitter` is the usual first splitter: it tries paragraphs, then sentences, then characters. Our docs are short, so most stay as a single chunk — the same API still works when files get long.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=280, chunk_overlap=40)
chunks = splitter.split_documents(raw_documents)
print(f"{len(chunks)} chunk(s):")
for c in chunks:
    print(f"  [{c.metadata['source']}] {c.page_content}")

## 3. Embed and store in Chroma

Chroma is an open-source vector database that runs locally — a good prototype default (Week 6 also covers FAISS and Pinecone). Embeddings run on your machine with MiniLM; no embedding API key.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="loop-labs-handbook",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Indexed {len(chunks)} chunk(s) into Chroma.")

## 4. Retrieve (no LLM yet)

Ask a question in plain language. Chroma returns the nearest chunks — the same semantic search you ranked by hand in Class 2.

In [ ]:
question = "How many vacation days do new hires get?"
hits = retriever.invoke(question)
for i, doc in enumerate(hits, 1):
    print(f"{i}. [{doc.metadata['source']}] {doc.page_content}")

## 5. Generate a grounded answer with Groq

Stuff the retrieved chunks into a prompt and ask Groq (`llama-3.3-70b-versatile`) to answer **only** from that context. That retrieve → prompt → generate loop is RAG.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_template(
    """Answer using ONLY the context below. If the answer is not in the context, say you don't know.

CONTEXT:
{context}

QUESTION: {question}
"""
)


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


def make_rag():
    if not GROQ_API_KEY:
        print("Skipping RAG chain — no GROQ_API_KEY set.")
        return None
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


rag = make_rag()
if rag is not None:
    print(rag.invoke(question))

Try a question the handbook does **not** cover — a good RAG prompt should refuse to invent an answer.

In [ ]:
if rag is None:
    print("Skipping — no GROQ_API_KEY set.")
else:
    print(rag.invoke("What is the pet-friendly office stipend?"))

## 6. Same chunks, different vector DB: FAISS

Chroma, FAISS, and Pinecone all store vectors and return nearest neighbors. Swap the store; keep the embeddings and the RAG prompt. FAISS is a local similarity-search library (you manage persistence). Pinecone is the hosted option — it needs its own API key, so we skip it here.

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 2})

print("FAISS hits for the same PTO question:")
for i, doc in enumerate(faiss_retriever.invoke(question), 1):
    print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:90]}...")

# Point the RAG chain at FAISS instead of Chroma (same prompt + Groq).
if GROQ_API_KEY:
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)
    faiss_rag = (
        {"context": faiss_retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("\nFAISS-grounded answer:")
    print(faiss_rag.invoke(question))
else:
    print("\nSkipping FAISS generation — no GROQ_API_KEY set.")

## 7. Retriever as an agent tool (optional)

Week 5's agent used tools like `calculator` and `mock_search`. A handbook retriever is the same pattern: a tool the model can call when the question is about policy. Chat memory (the message list) and document memory (the vector DB) solve different jobs — you often want both.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook. Use for vacation, expenses, and remote-work questions."""
    docs = retriever.invoke(query)
    if not docs:
        return "No handbook passages found."
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


def ask_handbook_agent(text: str):
    if not GROQ_API_KEY:
        print("Skipping agent — no GROQ_API_KEY set.")
        return None
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)
    agent = create_agent(
        model=llm,
        tools=[search_handbook],
        system_prompt=(
            "You are the employee handbook assistant. "
            "Call search_handbook before answering policy questions. "
            "If the tool result does not contain the answer, say you don't know."
        ),
    )
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"recursion_limit": 8},
    )
    content = result["messages"][-1].content
    print(content)
    return result


ask_handbook_agent("How many vacation days do new hires get?")

## Challenges

Each one builds on `chunks`, `embeddings`, `retriever`, and `rag` above.

### Challenge 1 — Add a fourth document

Add a short equipment policy (e.g. company laptops), re-run the splitter + Chroma index, and ask whether laptops are provided.

In [ ]:
# TODO: append a Document, rebuild vectorstore/retriever, ask a remote-work question

### Challenge 2 — Force a "don't know"

Ask `rag` something the three original documents never mention. Confirm the model refuses to guess.

In [ ]:
# TODO: rag.invoke(...) on an uncovered question

### Challenge 3 — Change k

Rebuild the retriever with `k=1` and `k=3`. Print the retrieved sources for the PTO question and note what extra (or missing) context does to the answer.

In [ ]:
# TODO: compare retriever search_kwargs k=1 vs k=3

### Challenge 4 (stretch) — Cite the source

Change the prompt so the answer includes the `source` filename inline (e.g. `(vacation.md)`).

In [ ]:
# TODO: update the prompt / format_docs and re-run rag.invoke